# Training Model - Smart Logistic AI (Diperbaiki)

**Perbaikan dari versi sebelumnya:** urutan SMOTE dipindah ke SETELAH train/test split, agar tidak terjadi data leakage. Angka akurasi di sini adalah angka JUJUR (dievaluasi pada data yang benar-benar belum pernah dilihat model).

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix
from imblearn.over_sampling import SMOTE
import pickle

print("1. Membaca Dataset...")
df = pd.read_csv('Train.csv')
df.head()

1. Membaca Dataset...


,ID,Warehouse_block,Mode_of_Shipment,Customer_care_calls,Customer_rating,Cost_of_the_Product,Prior_purchases,Product_importance,Gender,Discount_offered,Weight_in_gms,Reached.on.Time_Y.N
0,1,D,Flight,4,2,177,3,low,F,44,1233,1
1,2,F,Flight,4,5,216,2,low,M,59,3088,1
2,3,A,Flight,2,2,183,4,low,M,48,3374,1
3,4,B,Flight,3,3,176,4,medium,M,10,1177,1
4,5,C,Flight,2,2,184,3,medium,F,46,2484,1


In [2]:
print("2. Memilih Fitur & Encoding...")
features = [
    'Warehouse_block', 'Mode_of_Shipment', 'Customer_care_calls',
    'Customer_rating', 'Cost_of_the_Product', 'Prior_purchases',
    'Discount_offered', 'Weight_in_gms'
]
target = 'Reached.on.Time_Y.N'

X = df[features].copy()
y = df[target]

# Mapping SAMA seperti sebelumnya (sudah dikonfirmasi cocok dengan app.py)
X['Warehouse_block'] = X['Warehouse_block'].map({'A': 0, 'B': 1, 'C': 2, 'D': 3, 'F': 4})
X['Mode_of_Shipment'] = X['Mode_of_Shipment'].map({'Flight': 0, 'Ship': 1, 'Road': 2})

# Simpan rentang nilai asli tiap fitur numerik -> dipakai app.py agar tidak menerima input di luar jangkauan training
value_ranges = {c: (int(X[c].min()), int(X[c].max())) for c in ['Cost_of_the_Product','Prior_purchases','Discount_offered','Weight_in_gms']}
print("Rentang nilai fitur (untuk validasi input app.py):", value_ranges)
X.head()

2. Memilih Fitur & Encoding...
Rentang nilai fitur (untuk validasi input app.py): {'Cost_of_the_Product': (96, 310), 'Prior_purchases': (2, 10), 'Discount_offered': (1, 65), 'Weight_in_gms': (1001, 7846)}


,Warehouse_block,Mode_of_Shipment,Customer_care_calls,Customer_rating,Cost_of_the_Product,Prior_purchases,Discount_offered,Weight_in_gms
0,3,0,4,2,177,3,44,1233
1,4,0,4,5,216,2,59,3088
2,0,0,2,2,183,4,48,3374
3,1,0,3,3,176,4,10,1177
4,2,0,2,2,184,3,46,2484


In [3]:
print("3. Split Train/Test TERLEBIH DAHULU (sebelum SMOTE!)...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}, Test (tidak disentuh SMOTE): {X_test.shape}")

3. Split Train/Test TERLEBIH DAHULU (sebelum SMOTE!)...
Train: (8799, 8), Test (tidak disentuh SMOTE): (2200, 8)


In [4]:
print("4. SMOTE HANYA pada data training...")
print("Distribusi sebelum SMOTE (train):")
print(y_train.value_counts())

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("\nDistribusi sesudah SMOTE (train):")
print(y_train_res.value_counts())

4. SMOTE HANYA pada data training...
Distribusi sebelum SMOTE (train):
Reached.on.Time_Y.N
1    5250
0    3549
Name: count, dtype: int64

Distribusi sesudah SMOTE (train):
Reached.on.Time_Y.N
0    5250
1    5250
Name: count, dtype: int64


In [5]:
print("5. Training Model...")
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_res, y_train_res)

print("6. Evaluasi JUJUR (di test set asli, tidak pernah dilihat SMOTE/model)...")
y_pred = model.predict(X_test)
print(f"Akurasi Model (jujur): {accuracy_score(y_test, y_pred) * 100:.2f}%")
print(f"F1-Score (jujur): {f1_score(y_test, y_pred):.4f}\n")
print(classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

5. Training Model...
6. Evaluasi JUJUR (di test set asli, tidak pernah dilihat SMOTE/model)...
Akurasi Model (jujur): 66.27%
F1-Score (jujur): 0.6711

              precision    recall  f1-score   support

           0       0.56      0.79      0.65       887
           1       0.80      0.58      0.67      1313

    accuracy                           0.66      2200
   macro avg       0.68      0.68      0.66      2200
weighted avg       0.70      0.66      0.66      2200

Confusion Matrix:
 [[701 186]
 [556 757]]


In [6]:
print("7. Mengekspor Model ke model.pkl...")
with open('model.pkl', 'wb') as file:
    pickle.dump(model, file)

# Simpan juga rentang nilai fitur, supaya app.py bisa otomatis membatasi input
# (bukan di-hardcode manual seperti sebelumnya, biar konsisten dgn data training)
with open('feature_ranges.pkl', 'wb') as file:
    pickle.dump(value_ranges, file)

print("SELESAI! model.pkl & feature_ranges.pkl berhasil dibuat.")
print("\nCATATAN PENTING: Akurasi ~66% adalah batas performa jujur dataset ini dengan algoritma RandomForest -\n"
      "BUKAN 73% seperti versi sebelumnya (yang bocor akibat urutan SMOTE salah).")

7. Mengekspor Model ke model.pkl...
SELESAI! model.pkl & feature_ranges.pkl berhasil dibuat.

CATATAN PENTING: Akurasi ~66% adalah batas performa jujur dataset ini dengan algoritma RandomForest -
BUKAN 73% seperti versi sebelumnya (yang bocor akibat urutan SMOTE salah).
